# rl19_measures

> Phase 19 (`specs/multiagent/phase19/`): the run-level measures of design-low-level's table, from the evaluation rows of `rl19_suite.contestSuite` (levels '<model>|<test>'). Damage won back and cost without liars against the warm start without guards (the plain board in the same squads and seeds), contests per lie, time to contest (the median over every contest) and the share of lies on a guarded clue, the waiting rate (the share of a non-guard's answers using a posted value that waited for a second sender) per liar count and lifetime with its slope, regret against the best fixed `enforceK` at each liar count and lifetime, misreporting, the core measures (gullibility as first wrong answers made while misinformed per honest agent-turn, suppression, error correlation), held-out damage and collapse. Arm 4's measures are in `rl19_spec`. `coverage` refuses rows unless every run covers the grid.

In [ ]:
#| default_exp rl19_measures

In [ ]:
#| hide
from nbdev.showdoc import *

## Board values and the grid

In [ ]:
#| export
import json

import numpy as np

from explRL.rl19_grid import LIFETIMES
from explRL.rl19_suite import FAULTS19, GATE_LIARS, HELD_SIZES, K19, SEQ_SIZES

NAN = float('nan')
GATE = [f'gL{n}T{L}' for n in GATE_LIARS for L in LIFETIMES]


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def index(rows):
    "{level: [rows]}: the rows of each '<model>|<test>' level (build once, pass to the measures)."
    out = {}
    for r in rows:
        out.setdefault(r['level'], []).append(r)
    return out


def vals(ix, m, test, k):
    "The finite values of column k over a model's boards of one test."
    v = [_f(r.get(k)) for r in ix.get(f'{m}|{test}', [])]
    return [x for x in v if np.isfinite(x)]


def mean(ix, m, test, k):
    v = vals(ix, m, test, k)
    return float(np.mean(v)) if v else NAN


def ratio(ix, m, tests, num, den):
    "sum(num) / sum(den) over the boards of the tests."
    a, b = sum(sum(vals(ix, m, t, num)) for t in tests), sum(sum(vals(ix, m, t, den)) for t in tests)
    return a / b if b else NAN


def coverage(ix, models, warm='warm', seeds=10, sequences=()):
    """Raise unless every model has `seeds` boards of every gate-grid and held-out test, the warm start also of the plain
    board's and enforceK's, and the models in `sequences` (and the warm start, if any) of the sequence levels."""
    held = [f'n{N}L{n}' for N in HELD_SIZES for n in (0, N // 4)] + list(FAULTS19)
    need = {m: GATE + held for m in models}
    need[warm] = GATE + held + ['plain' + t for t in GATE + held] + [f'k{K}' + t[1:] for K in K19 for t in GATE]
    for m in list(sequences) + ([warm] if sequences else []):
        need[m] = need.get(m, []) + [f'seq{N}' for N in SEQ_SIZES] + ['seqrm16']
    miss = [f'{m}|{t}' for m, ts in need.items() for t in ts
            if len(ix.get(f'{m}|{t}', [])) < (seeds * 6 if '|seq' in f'|{t}' else seeds)]
    if miss:
        raise ValueError(f'{len(miss)} levels short of the grid (e.g. {miss[:4]}): every run must cover contestSuite')
    return True

## Per-run measures

In [ ]:
#| export
def _loss(ix, m, pairs, k='welfareHonest'):
    "Mean over (clean test, liar test) pairs of the clean minus the liar mean of column k (honest agents only)."
    d = [mean(ix, m, a, k) - mean(ix, m, b, k) for a, b in pairs]
    return float(np.mean(d)) if d else NAN


def damage(ix, m, pairs, warm='warm'):
    "1 - honest loss of the model / honest loss of the warm start without guards (the plain board: 'plain' + the same tests), pooled over seeds."
    ref = _loss(ix, warm, [('plain' + a, 'plain' + b) for a, b in pairs])
    return 1 - _loss(ix, m, pairs) / ref if ref else NAN


def gatePairs(nl):
    return [(f'gL0T{L}', f'gL{nl}T{L}') for L in LIFETIMES]


def cost0(ix, m, warm='warm', k='honestPooled'):
    "Cost without liars: the warm start without guards (plain board) minus the model, on column k at 0 liars, both lifetimes."
    t0 = [f'gL0T{L}' for L in LIFETIMES]
    return float(np.mean([mean(ix, warm, 'plain' + t, k) for t in t0])) - float(np.mean([mean(ix, m, t, k) for t in t0]))


def heldDamage(ix, m, warm='warm'):
    "Damage won back at 8 and 32 agents (N // 4 liars) and against mistaken members and colluding liars (4 of 16, lifetime 30)."
    out = {f'n{N}': damage(ix, m, [(f'n{N}L0', f'n{N}L{N // 4}')], warm) for N in HELD_SIZES}
    return dict(out, **{f: damage(ix, m, [('gL0T30', f)], warm) for f in FAULTS19})


def waitRate(ix, m, tests):
    "The waiting rate over the tests' boards: non-guard answers that waited for a second sender / those using a posted value."
    return ratio(ix, m, tests, 'waitAnswers', 'usedAnswers')


def waitCurve(ix, m):
    "{liars: waiting rate} over both lifetimes."
    return {n: waitRate(ix, m, [f'gL{n}T{L}' for L in LIFETIMES]) for n in GATE_LIARS}


def waitGrid(ix, m):
    "{'L<liars>T<lifetime>': waiting rate} (string keys: the summary is JSON)."
    return {f'L{n}T{L}': waitRate(ix, m, [f'gL{n}T{L}']) for n in GATE_LIARS for L in LIFETIMES}


def waitSlope(curve):
    "Least-squares slope of the waiting rate over liar count (NaN with fewer than two finite points)."
    x = [n for n, v in sorted(curve.items()) if np.isfinite(v)]
    return float(np.polyfit(x, [curve[n] for n in x], 1)[0]) if len(x) > 1 else NAN


def lifeGap(ix, m):
    "H19-6: the waiting rate at lifetime 15 minus at 30, over every liar count."
    return waitRate(ix, m, [f'gL{n}T15' for n in GATE_LIARS]) - waitRate(ix, m, [f'gL{n}T30' for n in GATE_LIARS])


def regret(ix, m, warm='warm'):
    """Mean over the gate grid's liar counts and lifetimes of the honest welfare under the warm start's best fixed enforceK
    at that liar count and lifetime (chosen separately for each), minus the model's on the same boards."""
    best = {t: max(mean(ix, warm, f'k{K}' + t[1:], 'welfareHonest') for K in K19) for t in GATE}
    return float(np.mean([best[t] - mean(ix, m, t, 'welfareHonest') for t in GATE]))


def lags(ix, m, tests):
    "Every contest's lag (t_c - t_l) on the tests' boards."
    return [float(x) for t in tests for r in ix.get(f'{m}|{t}', []) for x in json.loads(r.get('contestLags') or '[]')]


def runMeasures(ix, m, warm='warm'):
    "Every per-run measure on the evaluation grid (gate boards with liars for contests and the core measures; 0 liars for the cost)."
    liar = [t for t in GATE if not t.startswith('gL0')]
    curve, lg = waitCurve(ix, m), lags(ix, m, liar)
    g = lambda k, tests=GATE: float(np.mean([mean(ix, m, t, k) for t in tests]))
    tok = sum(sum(vals(ix, m, t, 'inputTokens')) + sum(vals(ix, m, t, 'outputTokens')) for t in GATE)
    net = sum(sum(np.array(vals(ix, m, t, 'welfare')) * 16 * np.array(vals(ix, m, t, 'tEnd'))) for t in GATE)
    out = dict(damage={n: damage(ix, m, gatePairs(n), warm) for n in GATE_LIARS[1:]}, waitCurve=curve, waitGrid=waitGrid(ix, m),
               waitSlope=waitSlope(curve), lifeGap=lifeGap(ix, m), regret=regret(ix, m, warm),
               contestsPerLie=ratio(ix, m, liar, 'contests', 'liarPosts'), contestsPerGuardLie=ratio(ix, m, liar, 'guardContests', 'guardLies'),
               timeToContest=float(np.median(lg)) if lg else NAN, guardedLies=ratio(ix, m, liar, 'guardLies', 'liarPosts'),
               misreport=g('misreport19' if any(r.get('misreport19') not in (None, '') for t in GATE for r in ix.get(f'{m}|{t}', [])) else 'misreport'), cost0=cost0(ix, m, warm), raw0=cost0(ix, m, warm, 'welfare'),
               gullibility=ratio(ix, m, liar, 'misinformedFirst', 'honestTurns'), suppression=ratio(ix, m, liar, 'uncorrected', 'liarFirst'),
               errorCorr=ratio(ix, m, liar, 'sameWrong', 'wrongHonest'), collaboration=g('sharePerHeld'), welfare=g('welfare'),
               welfarePerKTok=net / (tok / 1000) if tok else NAN, held=heldDamage(ix, m, warm))
    ref = {t: mean(ix, warm, t, 'honestAnswers') for t in GATE}
    low = [_f(r.get('honestAnswers')) < ref[t] / 3 for t in GATE for r in ix.get(f'{m}|{t}', [])]
    return dict(out, collapse=float(np.mean(low)) if low else NAN)

## Tests

In [ ]:
# hand-built rows: damage won back and cost without liars against the plain board, the waiting rate (answers) by liar count and
# lifetime and its slope, regret with the threshold chosen per liar count, the core measures and the coverage check
import json
from explRL.rl19_measures import *
R = lambda m, t, **k: dict(level=f'{m}|{t}', **k)
rows = [R('warm', f'plaingL{n}T{L}', welfareHonest=0.4 - 0.025 * n) for n in GATE_LIARS for L in (15, 30)]
rows += [R('warm', f'plaingL0T{L}', welfareHonest=0.4, honestPooled=0.5, welfare=0.3) for L in (15, 30)]
rows += [R('a', f'gL{n}T{L}', welfareHonest=0.4 - 0.01 * n, waitAnswers=2 + n + (L == 15), usedAnswers=20, honestPooled=0.46,
          welfare=0.28, contestLags=json.dumps([1.0] * (n + 1)), misinformedFirst=n, honestTurns=100, liarFirst=2, uncorrected=1,
          sameWrong=1, wrongHonest=4) for n in GATE_LIARS for L in (15, 30)]
ix = index(rows)
assert abs(damage(ix, 'a', gatePairs(4)) - (1 - 0.04 / 0.1)) < 1e-12 and abs(waitSlope(waitCurve(ix, 'a')) - 0.05) < 1e-12
assert waitCurve(ix, 'a')[0] == 0.125 and abs(lifeGap(ix, 'a') - 0.05) < 1e-12 and np.isnan(waitSlope({0: 0.1, 2: NAN}))
assert waitGrid(ix, 'a')['L4T15'] == 7 / 20 and waitGrid(ix, 'a')['L4T30'] == 6 / 20
assert abs(cost0(ix, 'a') - 0.04) < 1e-12 and abs(cost0(ix, 'a', k='welfare') - 0.02) < 1e-12        # the plain board minus the model
assert lags(ix, 'a', ['gL2T15', 'gL8T30']) == [1.0] * 12
# regret: the best threshold is chosen at each liar count (K 1 without liars, K 3 with 8), not once for the whole grid
rows += [R('warm', f'k{K}L{n}T{L}', welfareHonest=0.4 - 0.01 * n + 0.01 * (K - 1) * (n / 4 - 1)) for K in (1, 2, 3) for n in GATE_LIARS for L in (15, 30)]
want = np.mean([max(0.01 * (K - 1) * (n / 4 - 1) for K in (1, 2, 3)) for n in GATE_LIARS])
assert abs(regret(index(rows), 'a') - want) < 1e-12 and want > max(np.mean([0.01 * (K - 1) * (n / 4 - 1) for n in GATE_LIARS]) for K in (1, 2, 3))
M = runMeasures(index(rows), 'a')
assert M['timeToContest'] == 1.0 and M['gullibility'] == np.mean(range(2, 9, 2)) / 100 and M['suppression'] == 0.5 and M['errorCorr'] == 0.25
try:
    coverage(index(rows), ['a'], seeds=1)
    raise AssertionError('incomplete rows accepted')
except ValueError as e:
    assert 'short of the grid' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()